# Hybrid Arabic RAG — Simple Colab Version

هذه النسخة تعتمد مساراً بسيطاً وواضحاً:

```text
Document
   ↓
LlamaParse Cloud
OCR + Layout + Tables + Arabic/English parsing
   ↓
Page-aware Markdown
   ↓
──────────── Cloud ends here ────────────
   ↓
Local Chunking
   ↓
BGE-M3 Dense Embeddings
   +
BM25 Sparse Search
   ↓
Qdrant Local
   ↓
RRF Fusion
   ↓
BGE Reranker v2-m3
   ↓
Local Qwen3
   ↓
Answer + Sources
```

## لماذا هذه النسخة؟

- **LlamaParse فقط Cloud** لأنه مسؤول عن أصعب جزء: فهم الوثيقة واستخراجها.
- بقية الـRAG يعمل **Local داخل Colab**.
- لا يوجد MinerU.
- لا يوجد PaddleOCR.
- لا يوجد VLM OCR fallback.
- لا يوجد OCR quality heuristics.
- الكود قريب قدر الإمكان من نسخة الـCloud الأساسية.

> يفضّل تشغيل Colab باستخدام GPU.

In [ ]:
# ============================================================
# 1) Install packages
# ============================================================

!pip install -q -U \
    "llama-cloud>=2.8" \
    llama-index-core \
    "qdrant-client>=1.15.2" \
    rank-bm25 \
    "transformers>=4.51.0" \
    accelerate \
    sentencepiece \
    ipywidgets

print("✅ Packages installed")

## 2. LlamaParse API Key

الخدمة السحابية الوحيدة في هذه النسخة هي **LlamaParse**.

كل ما بعد الـParsing يعمل محلياً.

In [ ]:
# ============================================================
# 2) API key
# ============================================================

import os
from getpass import getpass

os.environ["LLAMA_CLOUD_API_KEY"] = getpass(
    "LLAMA_CLOUD_API_KEY: "
)

print("✅ LlamaParse key loaded")

In [ ]:
# ============================================================
# 3) Configuration
# ============================================================

COLLECTION_NAME = "hybrid_local_rag"

# Chunking
CHUNK_SIZE = 800
CHUNK_OVERLAP = 80

# Dense embeddings
EMBED_MODEL_NAME = "BAAI/bge-m3"
EMBED_DIM = 1024
EMBED_MAX_LENGTH = 1024

# Retrieval
DENSE_CANDIDATES = 12
SPARSE_CANDIDATES = 12
RRF_TOP_K = 12
RRF_K = 60

# Reranker
RERANK_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANK_TOP_N = 5
RERANK_MAX_LENGTH = 1024

# Local LLM
LOCAL_LLM_MODEL = "Qwen/Qwen3-1.7B"
MAX_NEW_TOKENS = 700

print("Embedding:", EMBED_MODEL_NAME)
print("Reranker:", RERANK_MODEL_NAME)
print("LLM:", LOCAL_LLM_MODEL)

## 4. Upload Document

ارفع ملفاً واحداً للاختبار.

الملف يُرسل مباشرة إلى LlamaParse بدون محاولة استخراج محلي مسبقاً.

In [ ]:
# ============================================================
# 4) Upload document
# ============================================================

from google.colab import files

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("❌ لم يتم رفع أي ملف.")

FILE_PATH = next(iter(uploaded.keys()))

print("✅ File:", FILE_PATH)

## 5. Cloud Parsing with LlamaParse

هذه هي **المرحلة السحابية الوحيدة**.

نطلب:

- Agentic parsing
- Arabic + English OCR
- Markdown
- Tables as Markdown
- Page-aware output

In [ ]:
# ============================================================
# 5) LlamaParse Cloud
# ============================================================

from llama_cloud import LlamaCloud

llama_client = LlamaCloud()

print("⏳ Uploading document to LlamaParse...")

cloud_file = llama_client.files.create(
    file=FILE_PATH,
    purpose="parse",
)

print("✅ File ID:", cloud_file.id)
print("⏳ Parsing document...")

parse_result = llama_client.parsing.parse(
    file_id=cloud_file.id,
    tier="agentic",
    version="latest",
    output_options={
        "markdown": {
            "tables": {
                "output_tables_as_markdown": True
            }
        }
    },
    processing_options={
        "ocr_parameters": {
            "languages": ["ar", "en"]
        }
    },
    expand=["markdown"],
)

pages = parse_result.markdown.pages

if not pages:
    raise RuntimeError(
        "❌ LlamaParse did not return pages."
    )

print(f"✅ Parsed pages: {len(pages)}")

print("\n--- First page preview ---\n")
print((pages[0].markdown or "")[:3000])

## 6. Chunking

بعد هذه النقطة انتهى دور الـCloud.

نحوّل كل صفحة إلى `Document` ونستخدم `SentenceSplitter` بنفس منطق النسخة الأساسية.

In [ ]:
# ============================================================
# 6) Documents + Chunking
# ============================================================

from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

documents = []

for page_number, page in enumerate(
    pages,
    start=1,
):
    text = (page.markdown or "").strip()

    if not text:
        continue

    documents.append(
        Document(
            text=text,
            metadata={
                "source": FILE_PATH,
                "page": page_number,
            },
        )
    )

if not documents:
    raise RuntimeError(
        "❌ No usable parsed text."
    )

splitter = SentenceSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

nodes = splitter.get_nodes_from_documents(
    documents
)

if not nodes:
    raise RuntimeError(
        "❌ Chunking produced no chunks."
    )

print("✅ Documents:", len(documents))
print("✅ Chunks:", len(nodes))

print("\n--- Example chunk ---\n")
print(nodes[0].text[:2000])
print("\nMetadata:", nodes[0].metadata)

## 7. Local BGE-M3 Embeddings

نستخدم **CLS pooling**:

```python
outputs.last_hidden_state[:, 0]
```

ثم `L2 normalization`.

In [ ]:
# ============================================================
# 7) Load local BGE-M3
# ============================================================

import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModel,
)

EMBED_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

embed_tokenizer = AutoTokenizer.from_pretrained(
    EMBED_MODEL_NAME
)

embed_model = AutoModel.from_pretrained(
    EMBED_MODEL_NAME,
    dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
    low_cpu_mem_usage=True,
)

embed_model = embed_model.to(
    EMBED_DEVICE
)

embed_model.eval()

print(
    "✅ BGE-M3 ready on:",
    EMBED_DEVICE
)

In [ ]:
# ============================================================
# 8) Dense embedding function
# ============================================================

import numpy as np

def encode_dense(
    texts,
    batch_size=4,
):
    all_vectors = []

    for start in range(
        0,
        len(texts),
        batch_size,
    ):
        batch = texts[
            start:start + batch_size
        ]

        inputs = embed_tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=EMBED_MAX_LENGTH,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(EMBED_DEVICE)
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            outputs = embed_model(
                **inputs
            )

            vectors = (
                outputs
                .last_hidden_state[:, 0]
            )

            vectors = F.normalize(
                vectors,
                p=2,
                dim=1,
            )

        all_vectors.append(
            vectors
            .cpu()
            .float()
            .numpy()
        )

    return np.concatenate(
        all_vectors,
        axis=0,
    )


chunk_texts = [
    node.text
    for node in nodes
]

print("⏳ Creating local embeddings...")

dense_vectors = encode_dense(
    chunk_texts
)

if dense_vectors.shape != (
    len(nodes),
    EMBED_DIM,
):
    raise RuntimeError(
        f"❌ Unexpected embedding shape: "
        f"{dense_vectors.shape}"
    )

print(
    "✅ Dense vectors:",
    dense_vectors.shape
)

## 8. Local Qdrant + Local BM25

لنبقي الكود بسيطاً:

- **Qdrant Local** يخزن الـDense vectors.
- `rank-bm25` ينفذ Sparse/BM25 محلياً.
- ندمج الترتيبين لاحقاً بـRRF.

لا نحتاج Qdrant Cloud ولا Sparse-vector configuration معقد.

In [ ]:
# ============================================================
# 9) Local Qdrant + BM25 index
# ============================================================

import re

from rank_bm25 import BM25Okapi
from qdrant_client import (
    QdrantClient,
    models,
)

# In-memory Qdrant for this Colab experiment
qdrant = QdrantClient(":memory:")

qdrant.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=models.VectorParams(
        size=EMBED_DIM,
        distance=models.Distance.COSINE,
    ),
)

points = []

# point id 1 -> nodes[0]
# point id 2 -> nodes[1]
node_by_id = {}

for point_id, (
    node,
    vector,
) in enumerate(
    zip(
        nodes,
        dense_vectors,
    ),
    start=1,
):
    node_by_id[point_id] = node

    points.append(
        models.PointStruct(
            id=point_id,
            vector=vector.tolist(),
            payload={
                "text": node.text,
                "source": node.metadata.get(
                    "source"
                ),
                "page": node.metadata.get(
                    "page"
                ),
                "chunk_index": point_id,
            },
        )
    )

qdrant.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

def bm25_tokenize(text):
    return re.findall(
        r"[\w\u0600-\u06FF]+",
        text.lower(),
    )

bm25_corpus = [
    bm25_tokenize(text)
    for text in chunk_texts
]

bm25 = BM25Okapi(
    bm25_corpus
)

print(
    f"✅ Qdrant Local points: {len(points)}"
)
print(
    f"✅ BM25 documents: {len(bm25_corpus)}"
)

## 9. Hybrid Retrieval

ننفذ:

1. Dense search من Qdrant.
2. BM25 search محلي.
3. RRF لدمج النتائج.

الكود متعمد أن يبقى بسيطاً وواضحاً.

In [ ]:
# ============================================================
# 10) Hybrid retrieval: Dense + BM25 + RRF
# ============================================================

def hybrid_retrieve(
    question,
    top_k=RRF_TOP_K,
):
    question = question.strip()

    if not question:
        return []

    # ---------------- Dense ----------------

    query_vector = encode_dense(
        [question],
        batch_size=1,
    )[0]

    dense_points = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector.tolist(),
        limit=min(
            DENSE_CANDIDATES,
            len(nodes),
        ),
        with_payload=True,
    ).points

    dense_ids = [
        int(point.id)
        for point in dense_points
    ]

    # ---------------- BM25 ----------------

    bm25_scores = bm25.get_scores(
        bm25_tokenize(question)
    )

    sparse_indices = np.argsort(
        bm25_scores
    )[::-1][
        :min(
            SPARSE_CANDIDATES,
            len(nodes),
        )
    ]

    sparse_ids = [
        int(index) + 1
        for index in sparse_indices
    ]

    # ---------------- RRF ----------------

    rrf_scores = {}

    for rank, point_id in enumerate(
        dense_ids,
        start=1,
    ):
        rrf_scores[point_id] = (
            rrf_scores.get(point_id, 0.0)
            + 1.0 / (RRF_K + rank)
        )

    for rank, point_id in enumerate(
        sparse_ids,
        start=1,
    ):
        rrf_scores[point_id] = (
            rrf_scores.get(point_id, 0.0)
            + 1.0 / (RRF_K + rank)
        )

    ranked_ids = sorted(
        rrf_scores,
        key=rrf_scores.get,
        reverse=True,
    )[:top_k]

    results = []

    for point_id in ranked_ids:
        node = node_by_id[point_id]

        results.append(
            {
                "id": point_id,
                "text": node.text,
                "source": node.metadata.get(
                    "source"
                ),
                "page": node.metadata.get(
                    "page"
                ),
                "rrf_score": rrf_scores[
                    point_id
                ],
            }
        )

    return results


def debug_retrieval(
    question,
    top_k=5,
):
    results = hybrid_retrieve(
        question,
        top_k=top_k,
    )

    for i, item in enumerate(
        results,
        start=1,
    ):
        print("=" * 80)
        print(
            f"#{i} | Page {item['page']} "
            f"| RRF={item['rrf_score']:.6f}"
        )
        print(item["text"][:1000])
        print()

    return results


print("✅ Hybrid retrieval ready")

## 10. Local BGE Reranker

In [ ]:
# ============================================================
# 11) Load local BGE reranker
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

RERANK_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

rerank_tokenizer = AutoTokenizer.from_pretrained(
    RERANK_MODEL_NAME
)

rerank_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        RERANK_MODEL_NAME,
        dtype=(
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        ),
        low_cpu_mem_usage=True,
    )
)

rerank_model = rerank_model.to(
    RERANK_DEVICE
)

rerank_model.eval()

print(
    "✅ Reranker ready on:",
    RERANK_DEVICE
)

In [ ]:
# ============================================================
# 12) Reranking function
# ============================================================

def rerank_results(
    question,
    candidates,
):
    if not candidates:
        return []

    pairs = [
        [
            question,
            item["text"],
        ]
        for item in candidates
    ]

    inputs = rerank_tokenizer(
        pairs,
        padding=True,
        truncation=True,
        max_length=RERANK_MAX_LENGTH,
        return_tensors="pt",
    )

    inputs = {
        key: value.to(
            RERANK_DEVICE
        )
        for key, value in inputs.items()
    }

    with torch.inference_mode():
        logits = (
            rerank_model(
                **inputs,
                return_dict=True,
            )
            .logits
            .view(-1)
            .float()
        )

    scores = torch.sigmoid(
        logits
    ).cpu().tolist()

    results = []

    for item, raw, score in zip(
        candidates,
        logits.cpu().tolist(),
        scores,
    ):
        result = dict(item)
        result["reranker_raw_score"] = float(raw)
        result["reranker_score"] = float(score)
        results.append(result)

    results.sort(
        key=lambda x: x[
            "reranker_raw_score"
        ],
        reverse=True,
    )

    return results[:RERANK_TOP_N]


print("✅ Reranking function ready")

## 11. Local Qwen3

In [ ]:
# ============================================================
# 13) Load local Qwen3
# ============================================================

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

LLM_DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

llm_tokenizer = AutoTokenizer.from_pretrained(
    LOCAL_LLM_MODEL
)

llm_model = (
    AutoModelForCausalLM
    .from_pretrained(
        LOCAL_LLM_MODEL,
        dtype=(
            torch.float16
            if torch.cuda.is_available()
            else torch.float32
        ),
        low_cpu_mem_usage=True,
    )
)

llm_model = llm_model.to(
    LLM_DEVICE
)

llm_model.eval()

print(
    "✅ Local Qwen ready on:",
    LLM_DEVICE
)

## 12. Prompt + Final RAG Function

In [ ]:
# ============================================================
# 14) Prompt + context
# ============================================================

SYSTEM_PROMPT = """
أنت مساعد عربي يعمل ضمن نظام RAG للإجابة عن الأسئلة اعتماداً على الوثائق المسترجعة.

التزم بالقواعد التالية:

1. استخدم السياق المرفق فقط.
2. لا تعتمد على معلومات خارجية.
3. إذا كان الجواب موجوداً في السياق، أجب مباشرة.
4. إذا لم توجد معلومات كافية، قل:
   لا توجد معلومات كافية في الوثائق للإجابة بدقة.
5. لا تخترع معلومات.
6. أشر إلى المصدر بصيغة [المصدر 1] أو [المصدر 2].
7. أجب بالعربية إلا إذا طلب المستخدم لغة أخرى.
""".strip()


def build_context(
    reranked_results,
):
    blocks = []

    for i, item in enumerate(
        reranked_results,
        start=1,
    ):
        blocks.append(
            f"[المصدر {i}]\n"
            f"الملف: {item['source']}\n"
            f"الصفحة: {item['page']}\n\n"
            f"{item['text']}"
        )

    return "\n\n".join(
        blocks
    )

In [ ]:
# ============================================================
# 15) ask_rag()
# ============================================================

def ask_rag(
    question,
    verbose=False,
):
    question = (question or "").strip()

    if not question:
        return {
            "answer": "الرجاء إدخال سؤال.",
            "sources": [],
        }

    # 1) Hybrid retrieval
    candidates = hybrid_retrieve(
        question
    )

    if not candidates:
        return {
            "answer": (
                "لم يتم العثور على مقاطع مناسبة."
            ),
            "sources": [],
        }

    # 2) Local reranking
    reranked = rerank_results(
        question,
        candidates,
    )

    # 3) Build context
    context = build_context(
        reranked
    )

    user_prompt = f"""
السؤال:
{question}

السياق المسترجع:
-----------------------------
{context}
-----------------------------

أجب عن السؤال اعتماداً على السياق فقط.
""".strip()

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    # 4) Local Qwen generation
    inputs = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        enable_thinking=False,
        return_dict=True,
        return_tensors="pt",
    ).to(LLM_DEVICE)

    input_length = inputs[
        "input_ids"
    ].shape[-1]

    with torch.inference_mode():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=0.7,
            top_p=0.8,
            top_k=20,
        )

    answer = llm_tokenizer.decode(
        outputs[0][input_length:],
        skip_special_tokens=True,
    ).strip()

    sources = []

    for i, item in enumerate(
        reranked,
        start=1,
    ):
        sources.append(
            {
                "source_number": i,
                "file": item["source"],
                "page": item["page"],
                "reranker_score":
                    item["reranker_score"],
                "preview":
                    item["text"][:350],
            }
        )

    if verbose:
        print("# Answer")
        print(answer)
        print()
        print("# Sources")

        for source in sources:
            print(
                f"[{source['source_number']}] "
                f"{source['file']} "
                f"- page {source['page']} "
                f"- score="
                f"{source['reranker_score']:.4f}"
            )

    return {
        "answer": answer,
        "sources": sources,
    }


print("✅ RAG pipeline ready")

# 13. Interactive Question Box

يمكنك طرح عدة أسئلة بدون إعادة تشغيل مراحل الـParsing والـEmbedding.

In [ ]:
# ============================================================
# 16) Interactive question box
# ============================================================

import ipywidgets as widgets

from IPython.display import (
    display,
    Markdown,
    clear_output,
)

question_box = widgets.Textarea(
    value="",
    placeholder="اكتب سؤالك عن الوثيقة هنا...",
    description="السؤال:",
    layout=widgets.Layout(
        width="100%",
        height="100px",
    ),
    style={
        "description_width": "70px"
    },
)

ask_button = widgets.Button(
    description="اسأل النظام",
    button_style="primary",
    icon="search",
)

debug_checkbox = widgets.Checkbox(
    value=False,
    description="إظهار المصادر",
)

output_box = widgets.Output()


def on_ask_clicked(_):
    question = question_box.value.strip()

    with output_box:
        clear_output(wait=True)

        if not question:
            display(
                Markdown(
                    "⚠️ **اكتب سؤالاً أولاً.**"
                )
            )
            return

        print(
            "⏳ Searching and generating..."
        )

        try:
            result = ask_rag(
                question
            )

            clear_output(wait=True)

            display(
                Markdown(
                    "## الإجابة\n\n"
                    + result["answer"]
                )
            )

            if (
                debug_checkbox.value
                and result["sources"]
            ):
                display(
                    Markdown(
                        "## المصادر"
                    )
                )

                for source in result[
                    "sources"
                ]:
                    display(
                        Markdown(
                            f"**[المصدر "
                            f"{source['source_number']}]**  \n"
                            f"- الملف: `{source['file']}`  \n"
                            f"- الصفحة: **{source['page']}**  \n"
                            f"- Reranker: "
                            f"`{source['reranker_score']:.4f}`  \n\n"
                            f"{source['preview']}..."
                        )
                    )

        except Exception as e:
            clear_output(wait=True)

            display(
                Markdown(
                    "## ❌ حدث خطأ"
                )
            )

            print(
                type(e).__name__,
                str(e),
            )


ask_button.on_click(
    on_ask_clicked
)

display(
    widgets.VBox(
        [
            question_box,
            widgets.HBox(
                [
                    ask_button,
                    debug_checkbox,
                ]
            ),
            output_box,
        ]
    )
)

# Final Flow

```text
Upload Document
      ↓
LlamaParse Cloud
      ↓
Page-aware Markdown
      ↓
SentenceSplitter
      ↓
BGE-M3 Local
      ↓
Qdrant Local
      +
BM25 Local
      ↓
RRF
      ↓
BGE Reranker Local
      ↓
Qwen3 Local
      ↓
Answer + Page Sources
```

## Cloud vs Local

**Cloud**
- LlamaParse فقط.

**Local**
- Chunking
- BGE-M3
- BM25
- Qdrant
- RRF
- BGE Reranker
- Qwen3
- Prompt
- Question UI